In [0]:
# https://docs.databricks.com/aws/en/mlflow/mlflow3-dl-workflow
# MLR 15.4 ? not liking 16x -- wrong PyTorch dependencies =S

# MLflow 3.0 deep learning example
This notebook first runs a model training job, which is tracked as an MLflow Run. It stores a model checkpoint every 10 epochs. Each checkpoint is tracked as an MLflow `LoggedModel`. You can then select the best checkpoint to deploy for production applications.

In [0]:
# Install the correct version of PyTorch
# %pip install torch==1.9.0

# Upgrade mlflow and install scikit-learn
%pip install mlflow --upgrade scikit-learn

# Restart the Python process to ensure the new packages are used
dbutils.library.restartPython()

In [0]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


In [0]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import pandas as pd
import torch
import torch.nn as nn
import mlflow
import mlflow.pytorch
from mlflow.entities import Dataset

# Helper function to prepare data
def prepare_data(df):
    X = torch.tensor(df.iloc[:, :-1].values, dtype=torch.float32)
    y = torch.tensor(df.iloc[:, -1].values, dtype=torch.long)
    return X, y

# Helper function to compute accuracy
def compute_accuracy(model, X, y):
    with torch.no_grad():
        outputs = model(X)
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y).sum().item() / y.size(0)
    return accuracy

# Define a basic PyTorch classifier
class IrisClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(IrisClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Load Iris dataset and prepare the DataFrame
iris = load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df['target'] = iris.target

# Split into training and testing datasets
train_df, test_df = train_test_split(iris_df, test_size=0.2, random_state=42)

# Prepare training data
train_dataset = mlflow.data.from_pandas(train_df, name="train")
X_train, y_train = prepare_data(train_dataset.df)

# Define the PyTorch model and move it to the device
input_size = X_train.shape[1]
hidden_size = 16
output_size = len(iris.target_names)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scripted_model = IrisClassifier(input_size, hidden_size, output_size).to(device)
scripted_model = torch.jit.script(scripted_model)

# Start a run to represent the training job
with mlflow.start_run():
    # Load the training dataset with MLflow. We will link training metrics to this dataset.
    train_dataset: Dataset = mlflow.data.from_pandas(train_df, name="train")
    X_train, y_train = prepare_data(train_dataset.df)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(scripted_model.parameters(), lr=0.01)

    for epoch in range(101):
        X_train, y_train = X_train.to(device), y_train.to(device)
        out = scripted_model(X_train)
        loss = criterion(out, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Log a checkpoint with metrics every 10 epochs
        if epoch % 10 == 0:
            # Each newly created LoggedModel checkpoint is linked with its
            # name, params, and step 
            model_info = mlflow.pytorch.log_model(
                pytorch_model=scripted_model,
                name=f"torch-iris-{epoch}",
                input_example=X_train.numpy(), ## 
            )
            # Log metric on training dataset at step and link to LoggedModel
            mlflow.log_metric(
                key="accuracy",
                value=compute_accuracy(scripted_model, X_train, y_train),
                step=epoch,
                model_id=model_info.model_id, ## ??
                dataset=train_dataset
            )

In [0]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import pandas as pd
import torch
import torch.nn as nn
import mlflow
import mlflow.pytorch

# Set the MLflow tracking URI to a supported scheme
mlflow.set_tracking_uri("databricks")

# Helper function to prepare data
def prepare_data(df):
    X = torch.tensor(df.iloc[:, :-1].values, dtype=torch.float32)
    y = torch.tensor(df.iloc[:, -1].values, dtype=torch.long)
    return X, y

# Helper function to compute accuracy
def compute_accuracy(model, X, y):
    with torch.no_grad():
        outputs = model(X)
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y).sum().item() / y.size(0)
    return accuracy

# Define a basic PyTorch classifier
class IrisClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(IrisClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        self.relu(x)
        x = self.fc2(x)
        return x

# Load Iris dataset and prepare the DataFrame
iris = load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df['target'] = iris.target

# Split into training and testing datasets
train_df, test_df = train_test_split(iris_df, test_size=0.2, random_state=42)

# Prepare training data
X_train, y_train = prepare_data(train_df)

# Define the PyTorch model and move it to the device
input_size = X_train.shape[1]
hidden_size = 16
output_size = len(iris.target_names)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scripted_model = IrisClassifier(input_size, hidden_size, output_size).to(device)
scripted_model = torch.jit.script(scripted_model)

# Start a run to represent the training job
with mlflow.start_run():
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(scripted_model.parameters(), lr=0.01)

    for epoch in range(101):
        X_train, y_train = X_train.to(device), y_train.to(device)
        out = scripted_model(X_train)
        loss = criterion(out, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Log a checkpoint with metrics every 10 epochs
        if epoch % 10 == 0:
            model_info = mlflow.pytorch.log_model(
                pytorch_model=scripted_model,
                artifact_path=f"torch-iris", #NOT QUITE RIGHT# complaining that it needed an artifact_path...?
                name=f"torch-iris-{epoch}", ## complaining that API is not installed?
                input_example=X_train.cpu().numpy(), ## this without .cpu() yieled errors 
            )
            mlflow.log_metric(
                key="accuracy",
                value=compute_accuracy(scripted_model, X_train, y_train),
                step=epoch
            )

This example produced one MLflow Run (`training_run`) and 11 MLflow Logged Models, one for each checkpoint (at steps 0, 10, …, 100). Using MLflow’s UI or search API, you can get the checkpoints and rank them by their accuracy.

In [0]:
# ranked_checkpoints = mlflow.search_logged_models(output_format="list")
# ranked_checkpoints.sort(
#     key=lambda model: next((metric.value for metric in model.metrics if metric.key == "accuracy"), float('-inf')),
#     reverse=True
# )

# best_checkpoint: mlflow.entities.LoggedModel = ranked_checkpoints[0]
# print(best_checkpoint.metrics[0])

In [0]:
from mlflow.tracking import MlflowClient
from mlflow.exceptions import RestException

client = MlflowClient()
ranked_checkpoints = client.search_registered_models()

def get_accuracy(model_version):
    run_id = model_version.run_id
    if run_id is None:
        return float('-inf')
    try:
        metrics = client.get_run(run_id).data.metrics
        return metrics.get("accuracy", float('-inf'))
    except RestException:
        return float('-inf')

ranked_checkpoints = [
    model for model in ranked_checkpoints if model.latest_versions
]

ranked_checkpoints.sort(
    key=lambda model: get_accuracy(model.latest_versions[0]),
    reverse=True
)

if ranked_checkpoints:
    best_checkpoint = ranked_checkpoints[0]
    best_run_id = best_checkpoint.latest_versions[0].run_id
    best_metrics = client.get_run(best_run_id).data.metrics
    print(best_metrics.get("accuracy"))
else:
    print("No valid checkpoints found.")

In [0]:
best_checkpoint

In [0]:
# best_checkpoint.model_uri
best_checkpoint.latest_versions[0].source

In [0]:
# best_checkpoint.latest_versions[0]  -- this is weird -- it's not me

In [0]:
worst_checkpoint: mlflow.entities.LoggedModel = ranked_checkpoints[-1]
print(worst_checkpoint.metrics)

In [0]:
import mlflow

# Assuming ranked_checkpoints contains RegisteredModel objects
worst_checkpoint = ranked_checkpoints[-1]
worst_checkpoint_run_id = worst_checkpoint.latest_versions[0].run_id
worst_checkpoint_run = mlflow.get_run(worst_checkpoint_run_id)
print(worst_checkpoint_run.data.metrics)

In [0]:
worst_checkpoint_run.data.metrics

After selecting the best checkpoint model, register that model to the model registry. You can also see the model ID, parameters, and metrics on the model version page in Catalog Explorer.

In [0]:
# Set the registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# You must have `USE CATALOG` privileges on the catalog, and you must have `USE SCHEMA` privileges on the schema.
# If necessary, change the catalog and schema name here.

CATALOG = "mmt" #"main"
SCHEMA = "mlflow_v3_assessbrickready" #"default"
MODEL = "dl_model"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.{MODEL}"

## uc_model_version = mlflow.register_model(f"models:/{best_checkpoint.model_uri}", name=MODEL_NAME)
# yields error: AttributeError: 'RegisteredModel' object has no attribute 'model_uri'

## update 
model_uri = best_checkpoint.latest_versions[0].source
uc_model_version = mlflow.register_model(f"models:/{model_uri}", name=MODEL_NAME)


Now you can view the model version and all centralized performance data on the model version page in Unity Catalog. You can also get the same information using the API as shown in the following cell.

In [0]:
# Get the model version
from mlflow import MlflowClient
client = MlflowClient()
model_version = client.get_model_version(name=MODEL_NAME, version=uc_model_version.version)
print(model_version)